# Clef-Flash on Colab → a Plexus `decisions` upstream — **v5**

**v5 changes.** Teardown now refuses to destroy a still-routed node, the self-test validates answer shape, and duplicate cleanup refuses to delete a freshly-seen node. Prior: **v4** replaced the fallback `log()` ordering bug. The notebook now uses `TAILSCALE_API_KEY` instead of a pre-minted `TS_AUTHKEY`.
Before joining it deletes every stale `clef-llama-remote` / `clef-llama-remote-<digits>` device
through the Tailscale API and re-reads the device list to prove none remain; it then mints a
single-use, ephemeral, preauthorised key for this join, revokes that key, and deletes its own
device again on teardown. `PLEXUS_API_KEY` is the notebook-scoped key `clef-llama-colab-v2`
(`allowRawPassthrough: true`; allowed providers `clef-llama-remote` and `clef-flash-mac`), and
the self-test calls `direct/clef-llama-remote/clef-flash` — the direct provider route, not the
alias selector. The v1 notebook is retained beside this one for reference.

Turns a **Colab GPU runtime** into an ephemeral Plexus upstream for the `cloudflare/clef-flash`
alias, and self-registers it. Nothing is baked into an image: every run rebuilds the runtime from
public releases.

**What a run does**

1. serves `bartowski/Cloudflare_clef-flash-GGUF` (Q4_K_M, 6.04 GB) with **llama.cpp `b12292`** —
   that release ships `POST /v1/systemone`, the System One / `decisions` surface;
2. joins the tailnet as an **ephemeral, preauthorised** node named `clef-llama-remote`
   (userspace networking: Colab has no `/dev/net/tun` and no systemd, so `install.sh` is not used);
3. puts a **round-robin, bearer-checking proxy** in front of one or two `llama-server` workers;
4. rewrites `clef-llama-remote.api_base_url.systemone` to this VM's **tailnet IPv4**
   (`*.ts.net` does not resolve from the Plexus host — CorpDNS/MagicDNS are off there);
5. **enables the rung** in the alias's `LOCAL` group, only once three things are proven: the
   workers answer `/health`, every worker is holding GPU memory (an `-ngl 99` request is not a
   result — `nvidia-smi --query-compute-apps` is the evidence), and the published port answers
   over the tailnet;
6. runs the required 2-question `decisions` self-test through `direct/clef-llama-remote/clef-flash`;
   a wrong rung or an empty answer tears the node down (rung disabled first).

Teardown disables the rung **first** (so Plexus stops routing here), restores the provider's
configured `maxConcurrency`, then kills the workers, then leaves the tailnet and deletes this run's Tailscale device. Teardown only runs
while the kernel is alive: **if Colab deletes or evicts the runtime, no callback runs** and the
rung stays enabled until someone disables it or Plexus cools the provider down.

## Hardware and session limits

| Requirement | Detail |
|---|---|
| GPU runtime | **required** — `Runtime → Change runtime type → T4 GPU` |
| GPUs | 2× T4 recommended (two workers); 1× T4 works (one worker); VRAM < 14 GiB is refused |
| Download | ~8 GB (llama.cpp 198 MB + CUDA runtime 594 MB + GGUF 6.04 GB) into `/content` |
| Session cap | ~12 h on Colab Pro; free tier is shorter |
| Idle | ~90 min with no browser tab → disconnect; closing the tab ends the VM |
| Colab Pro | strongly recommended — a free runtime is not a dependable 24×7 upstream |

Everything lives in the VM's `/content`, which is wiped with the VM. The ephemeral tailnet
registration goes with it; the next run re-registers and re-enables the rung.

## Context per GPU (KV cache: 32 KiB/token)

| VRAM | `-c` | KV (f16) | Notes |
|---|---|---|---|
| ≥ 22 GiB | 262144 | 8.00 GiB | full trained context |
| 14–22 GiB (T4) | 131072 | 4.00 GiB | 6.04 GB weights + 4 GiB KV ⇒ ~4 GiB headroom |
| < 14 GiB | — | — | the notebook refuses to start |

Override with `CLEF_CTX`; hard cap 262144. The value actually in force is read back from
`GET /props` → `default_generation_settings.n_ctx` and printed per worker.

## Secrets (Colab key panel, or the environment when run outside Colab)

| Name | Required | Used for |
|---|---|---|
| `TAILSCALE_API_KEY` | yes | Tailscale REST API: delete duplicate nodes, mint the per-run ephemeral auth key, resolve this run's device id, delete the node on teardown |
| `PLEXUS_ADMIN_KEY` | yes | `x-admin-key` on the Plexus management API (provider/alias writes require admin) |
| `CLEF_REMOTE_KEY` | yes | the bearer Plexus sends; the proxy and every worker require it |
| `PLEXUS_API_KEY` | yes | limited key `clef-llama-colab-v2`: direct-provider inference and the end-to-end self-test |

There is **no `TS_AUTHKEY`** in v2. Each run mints a single-use, ephemeral, preauthorised
`tag:endpoints` auth key through the API and revokes it immediately after `tailscale up`. The API
token itself never becomes a node credential, so the only standing Tailscale secret is
`TAILSCALE_API_KEY`.

Cells read `google.colab.userdata` and fall back to `os.environ`, so the same code runs outside
Colab. **No secret value is ever printed** — only its name and length.

`CLEF_REMOTE_KEY` must equal the `api_key` stored on the `clef-llama-remote` provider. Rotate both
together or every request 401s. Note that the key lands in the `llama-server`/`tailscale`
command lines, so it is visible to `ps` inside the VM — acceptable on a single-tenant ephemeral
VM, not on a shared host.

`PLEXUS_ADMIN_KEY` is the existing broad Plexus administration credential — provider/alias writes
are admin-only, so it cannot be replaced by the limited inference key. `PLEXUS_API_KEY` is
`clef-llama-colab-v2`, scoped to `clef-llama-remote` and `clef-flash-mac` with passthrough enabled;
keep the two distinct and treat the runtime as single-tenant.

In [ ]:
%%bash
# [shell 1/3] python deps. torch/CUDA already ship with a GPU runtime; do not reinstall them.
set -euo pipefail
python3 -m pip install --quiet --upgrade "huggingface_hub>=0.23.0"
python3 - <<'PY'
import sys, huggingface_hub
print("python         ", sys.version.split()[0])
print("huggingface_hub", huggingface_hub.__version__)
try:
    import torch
    print("torch          ", torch.__version__, "cuda available:", torch.cuda.is_available())
except Exception as exc:
    print("torch           not importable:", exc)
PY

In [ ]:
# [2/13] configuration: secrets, paths, constants
from __future__ import annotations

import os
import subprocess
import time
from pathlib import Path


# --------------------------------------------------------------------- secrets
def _secret(name: str, required: bool = True) -> str:
    """Colab userdata first, environment second. Never echoed."""
    value = ""
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        try:
            value = userdata.get(name) or ""
        except Exception:
            value = ""
    except Exception:
        value = ""
    if not value:
        value = os.environ.get(name, "")
    value = (value or "").strip()
    if required and not value:
        raise SystemExit(
            f"missing secret {name}: add it in the Colab key panel, or export it when running locally"
        )
    return value


SECRETS = {
    "TAILSCALE_API_KEY": _secret("TAILSCALE_API_KEY"),
    "PLEXUS_ADMIN_KEY": _secret("PLEXUS_ADMIN_KEY"),
    "CLEF_REMOTE_KEY": _secret("CLEF_REMOTE_KEY", required=False),  # optional; provider api_key is authoritative
    "PLEXUS_API_KEY": _secret("PLEXUS_API_KEY"),
}
print("secrets:")
for _name, _value in SECRETS.items():
    print(f"  {_name:<17} {'set  ' if _value else 'unset'} len={len(_value)}")


# ----------------------------------------------------------------------- paths
def _pick_root() -> Path:
    content = Path("/content")
    try:
        content.mkdir(parents=True, exist_ok=True)
    except Exception:
        pass
    if content.is_dir() and os.access(content, os.W_OK):
        return content
    fallback = Path(os.path.expanduser("~/.clef-colab"))
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


ROOT = Path(os.environ.get("CLEF_ROOT") or _pick_root())
LOG_DIR = ROOT / "logs"
MODEL_DIR = ROOT / "models"
LLAMA_ROOT = ROOT / "llama"
STATE_DIR = ROOT / "state"
for _dir in (LOG_DIR, MODEL_DIR, LLAMA_ROOT, STATE_DIR):
    _dir.mkdir(parents=True, exist_ok=True)


def _pick_ts_dir() -> Path:
    override = os.environ.get("CLEF_TS_PREFIX")
    candidates = [Path(override)] if override else [Path("/opt/tailscale"), ROOT / "tailscale"]
    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
        except Exception:
            continue
        if os.access(candidate, os.W_OK):
            return candidate
    raise SystemExit("no writable directory for the tailscale binaries")


TS_DIR = _pick_ts_dir()
TS_SOCK = Path("/var/run/tailscale/tailscaled.sock")
if not os.access(TS_SOCK.parent, os.W_OK):
    TS_SOCK = STATE_DIR / "tailscaled.sock"
TS_STATE = Path("/var/lib/tailscale/tailscaled.state")
if not os.access(TS_STATE.parent, os.W_OK):
    TS_STATE = STATE_DIR / "tailscaled.state"


# ------------------------------------------------------------------- constants
PLEXUS_BASE = (os.environ.get("PLEXUS_BASE") or "https://plexus.example.ts.net").rstrip("/")
ALIAS = "cloudflare/clef-flash"
PROVIDER = "clef-llama-remote"
MODEL = "clef-flash"
SELF_TEST_MODEL = f"direct/{PROVIDER}/{MODEL}"
TS_API_BASE = "https://api.tailscale.com/api/v2"
TS_TAG = "tag:endpoints"
TS_AUTHKEY_EXPIRY_SECONDS = 3600
TS_HOSTNAME = "clef-llama-remote"
EXPECTED_DNSNAME = os.environ.get("EXPECTED_DNSNAME", "").strip()


# The Colab secret is optional: the provider's api_key is the single source of truth, and the
# notebook already holds an admin key. Adopting it here keeps the proxy bearer and Plexus in
# lockstep even when the notebook-scoped CLEF_REMOTE_KEY secret is stale or unbound.
def _provider_api_key() -> str:
    import json as _json
    import urllib.request as _url

    request = _url.Request(
        f"{PLEXUS_BASE}/v0/management/providers/{PROVIDER}",
        headers={"x-admin-key": SECRETS["PLEXUS_ADMIN_KEY"]},
    )
    with _url.urlopen(request, timeout=30) as response:
        return (_json.load(response).get("api_key") or "").strip()


if not SECRETS.get("CLEF_REMOTE_KEY"):
    SECRETS["CLEF_REMOTE_KEY"] = _provider_api_key()
    if not SECRETS["CLEF_REMOTE_KEY"]:
        raise SystemExit("missing CLEF_REMOTE_KEY and the Plexus provider has no api_key")
    print(f"clef_remote_key: adopted provider api_key len={len(SECRETS['CLEF_REMOTE_KEY'])}")

GGUF_REPO = "bartowski/Cloudflare_clef-flash-GGUF"
GGUF_FILE = "Cloudflare_clef-flash-Q4_K_M.gguf"
GGUF_EXPECT_BYTES = 6_040_541_344
MMPROJ_FILE = "mmproj-Cloudflare_clef-flash-f16.gguf"
USE_MMPROJ = os.environ.get("CLEF_MMPROJ", "0").strip().lower() in ("1", "true", "yes", "on")

CTX_FULL = 262_144
CTX_T4 = 131_072
VRAM_HIGH_GIB = 22.0
VRAM_MIN_GIB = 14.0
KV_BYTES_PER_TOKEN = 32_768  # 8 full-attn layers * 4 kv heads * (256+256) * 2 B
CTX_OVERRIDE = os.environ.get("CLEF_CTX", "").strip()

WORKER_SPECS = ((0, 8091), (1, 8092))  # (CUDA_VISIBLE_DEVICES, loopback port)
SERVE_PORT_LADDER = (8080, 8443)
# The provider is created with timeoutMs=600000 (plexus_setup.py). The proxy gives up first, so
# its own 502 reaches Plexus before Plexus abandons the request. See README "Timeouts".
UPSTREAM_TIMEOUT = 540.0
READY_TIMEOUT = 900.0
SELF_TEST_TIMEOUT = 600.0
PROVIDER_TIMEOUT_MS = 600_000  # what the provider must allow; checked live at bring-up
MIN_PROVIDER_TIMEOUT_MS = 300_000  # below this a T4 pre-fill is cut off mid-request
PROBE_TIMEOUT = 20.0
WATCH_INTERVAL = 30.0
GPU_MIN_USED_MIB = 2048  # under this, a "GPU" load is really a CPU load with extra steps


def log(message: str) -> None:
    """Timestamped console line, mirrored into ROOT/logs/notebook.log."""
    print(f"{time.strftime('%H:%M:%S')} {message}", flush=True)
    try:
        with open(LOG_DIR / "notebook.log", "a") as handle:
            handle.write(f"{time.strftime('%Y-%m-%dT%H:%M:%S')} {message}\n")
    except Exception:
        pass


log(f"root={ROOT}  ts_dir={TS_DIR}  plexus={PLEXUS_BASE}")
log(f"mmproj={'on' if USE_MMPROJ else 'off'}  ctx_override={CTX_OVERRIDE or '-'}")
log(f"tailscaled state={TS_STATE} socket={TS_SOCK}")

In [ ]:
%%bash
# [shell 2/3] llama.cpp b12292 prebuilt CUDA release + its CUDA runtime libs.
# Built binaries only - no source build. Both tarballs extract flat into /content/llama.
set -euo pipefail
ROOT="${CLEF_ROOT:-/content}"
[ -d "$ROOT" ] && [ -w "$ROOT" ] || ROOT="$HOME/.clef-colab"
mkdir -p "$ROOT/llama" "$ROOT/logs"
cd "$ROOT/llama"

BASE="https://github.com/ggml-org/llama.cpp/releases/download/b12292"
# Archive roots are not asset names: the binary asset unpacks into llama-b12292/, so the old
# `-d "$ASSET"` guard never matched and re-downloaded 198 MB on every run. Guard on the payload.
BIN_ASSET="llama-b12292-bin-ubuntu-cuda-12.8-x64"
BIN_DIR="llama-b12292"
CUDART_ASSET="cudart-llama-b12292-bin-ubuntu-cuda-12.8-x64"
CUDART_DIR="cudart-llama-b12292-bin-ubuntu-cuda-12.8-x64"

if [ -x "$BIN_DIR/llama-server" ] && [ -f "$BIN_DIR/libggml-cuda.so" ]; then
  echo "skip $BIN_ASSET (already extracted in $BIN_DIR)"
else
  echo "+ $BIN_ASSET.tar.gz"
  curl -fL --retry 3 --retry-delay 2 --progress-bar -o "$BIN_ASSET.tar.gz" "$BASE/$BIN_ASSET.tar.gz"
  tar -xzf "$BIN_ASSET.tar.gz"
  rm -f "$BIN_ASSET.tar.gz"
fi

if [ -f "$CUDART_DIR/libcudart.so.12" ]; then
  echo "skip $CUDART_ASSET (already extracted in $CUDART_DIR)"
else
  echo "+ $CUDART_ASSET.tar.gz"
  curl -fL --retry 3 --retry-delay 2 --progress-bar -o "$CUDART_ASSET.tar.gz" "$BASE/$CUDART_ASSET.tar.gz"
  tar -xzf "$CUDART_ASSET.tar.gz"
  rm -f "$CUDART_ASSET.tar.gz"
fi

chmod +x "$ROOT"/llama/llama-b*/llama-* 2>/dev/null || true
echo
echo "llama-server binaries:"
find "$ROOT/llama" -maxdepth 2 -name 'llama-server' -type f -print -exec ls -lh {} \;
echo
du -sh "$ROOT/llama"/* 2>/dev/null || true

In [ ]:
# [4/13] locate the binaries, prove /v1/systemone exists, fetch the GGUF
from __future__ import annotations

import os
import shutil
from pathlib import Path

# --- 4.1 llama-server -------------------------------------------------------
servers = [
    path
    for path in sorted(LLAMA_ROOT.rglob("llama-server"))
    if path.is_file() and os.access(path, os.X_OK)
]
if not servers:
    raise SystemExit(
        f"no executable llama-server under {LLAMA_ROOT} — run the llama.cpp release cell ([shell 2/3]) first"
    )
LLAMA_SERVER = servers[0]
CUDART_DIRS = [
    path for path in sorted(LLAMA_ROOT.iterdir()) if path.is_dir() and path.name.startswith("cudart")
]
LIB_PATH = os.pathsep.join([str(path) for path in CUDART_DIRS] + [str(LLAMA_SERVER.parent)])
log(f"llama-server : {LLAMA_SERVER}")
log(f"cudart dirs  : {[path.name for path in CUDART_DIRS] or 'none'}")


# --- 4.2 hard capability gate ----------------------------------------------
def _contains(path: Path, needle: bytes, chunk: int = 8 << 20) -> bool:
    """Streaming substring search over a large shared object."""
    carry = b""
    with open(path, "rb") as handle:
        while True:
            block = handle.read(chunk)
            if not block:
                return False
            haystack = carry + block
            if needle in haystack:
                return True
            carry = haystack[-len(needle) :]


impl_candidates = [path for path in sorted(LLAMA_ROOT.rglob("libllama-server-impl.so*")) if path.is_file()]
if not impl_candidates:
    raise SystemExit(f"libllama-server-impl.so not found under {LLAMA_ROOT}")
impl = next((path for path in impl_candidates if _contains(path, b"/v1/systemone")), None)
if impl is None:
    raise SystemExit(
        "ABORT: this llama.cpp build has no /v1/systemone. The `decisions` surface would be missing "
        "and every Plexus call would 404. Check that [shell 2/3] pulled release b12292."
    )
log(f"systemone    : OK in {impl.name}")
libllama = next((path for path in sorted(LLAMA_ROOT.rglob("libllama.so*")) if path.is_file()), None)
if libllama is not None:
    marker = "present" if _contains(libllama, b"clef") else "ABSENT"
    log(f"clef arch    : {marker} string in {libllama.name} (sanity only, not a gate)")

# --- 4.3 model --------------------------------------------------------------
try:
    from huggingface_hub import hf_hub_download
except ImportError as exc:  # pragma: no cover - only when cell 1 was skipped
    raise SystemExit("huggingface_hub is missing — run the pip cell ([shell 1/3])") from exc

log(f"downloading  : {GGUF_REPO}/{GGUF_FILE} (~6.04 GB on first run)")
GGUF = Path(hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE, cache_dir=str(MODEL_DIR)))
GGUF_BYTES = GGUF.stat().st_size
log(f"gguf         : {GGUF} ({GGUF_BYTES / 1e9:.2f} GB)")
if GGUF_BYTES != GGUF_EXPECT_BYTES:
    log(
        f"WARNING: gguf is {GGUF_BYTES} bytes, expected {GGUF_EXPECT_BYTES}; "
        "continuing (the HF file may have been re-uploaded)"
    )

MMPROJ = None
if USE_MMPROJ:
    MMPROJ = Path(hf_hub_download(repo_id=GGUF_REPO, filename=MMPROJ_FILE, cache_dir=str(MODEL_DIR)))
    log(f"mmproj       : {MMPROJ} ({MMPROJ.stat().st_size / 1e9:.2f} GB)")
else:
    log("mmproj       : off (CLEF_MMPROJ=1 to enable image input, costs VRAM)")

free_gb = shutil.disk_usage(ROOT).free / 1e9
log(f"disk free    : {free_gb:.1f} GB")
if free_gb < 2:
    log("WARNING: less than 2 GB free — llama-server stdout will fail once logs grow")

In [ ]:
# [5/13] GPU inventory, context policy, worker plan
from __future__ import annotations

import re
import shutil
import subprocess


def _nvidia_smi() -> list:
    exe = shutil.which("nvidia-smi") or "/usr/bin/nvidia-smi"
    proc = subprocess.run(
        [exe, "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if proc.returncode != 0:
        raise SystemExit(f"nvidia-smi failed rc={proc.returncode}: {proc.stderr.strip()[:400]}")
    found = []
    for line in proc.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 3:
            continue
        index = int(parts[0])
        name = ",".join(parts[1:-1])
        number, _, unit = parts[-1].partition(" ")
        gib = float(number) / 1024.0 if unit.upper().startswith("MI") else float(number)
        found.append({"index": index, "name": name, "gib": round(gib, 2), "raw": parts[-1]})
    return found


GPUS = _nvidia_smi()
if not GPUS:
    raise SystemExit(
        "no GPUs visible: Runtime → Change runtime type → T4 GPU (2× T4 recommended), then re-run"
    )
log(f"gpus         : {len(GPUS)}")
for gpu in GPUS:
    log(f"  cuda:{gpu['index']} {gpu['name']} {gpu['gib']:.2f} GiB")


def gpu_compute_memory() -> dict:
    """pid -> MiB held on a GPU, from nvidia-smi's compute-apps view.

    This is the evidence of record for "the weights are on the card": `-ngl 99` is a request,
    and neither a command line nor a log line proves it was honoured. Returns {} when the
    driver declines to enumerate compute processes.
    """
    exe = shutil.which("nvidia-smi") or "/usr/bin/nvidia-smi"
    try:
        proc = subprocess.run(
            [exe, "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=120,
        )
    except (OSError, subprocess.TimeoutExpired):
        return {}
    if proc.returncode != 0:
        return {}
    held = {}
    for line in proc.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 2 or not parts[0].isdigit():
            continue
        try:
            held[int(parts[0])] = int(float(parts[1]))
        except ValueError:
            continue
    return held


_OFFLOAD = re.compile(r"offload\w*[^\n]*?(\d+)\s*/\s*(\d+)\s+layers?", re.IGNORECASE)


def gpu_layers_in_log(path) -> tuple | None:
    """(layers_on_gpu, total_layers) from llama.cpp's load log, or None if absent.

    Deliberately loose: the wording drifts between releases ("offloaded 33/33 layers to GPU",
    "offloading 32 repeating layers"). Only consulted when nvidia-smi cannot enumerate compute
    processes, so a missing line means "unproven", never "assume GPU".
    """
    try:
        with open(path, "r", errors="replace") as handle:
            text = handle.read()
    except OSError:
        return None
    matches = _OFFLOAD.findall(text)
    if not matches:
        return None
    on_gpu, total = matches[-1]
    return int(on_gpu), int(total)


def ctx_for(gib: float) -> int:
    """Context for one GPU, from that GPU's total VRAM."""
    if CTX_OVERRIDE:
        try:
            wanted = int(CTX_OVERRIDE)
        except ValueError:
            raise SystemExit(f"CLEF_CTX={CTX_OVERRIDE!r} is not an integer") from None
    elif gib >= VRAM_HIGH_GIB:
        wanted = CTX_FULL
    elif gib >= VRAM_MIN_GIB:
        wanted = CTX_T4
    else:
        raise SystemExit(
            f"{gib:.2f} GiB VRAM is below the {VRAM_MIN_GIB:.0f} GiB floor: the 6.04 GB model leaves "
            "no room for a usable KV cache. Use a T4 or larger runtime."
        )
    return min(wanted, CTX_FULL)


# GPU policy: 1 GPU -> one worker pinned to cuda:0; 2+ GPUs -> one worker per GPU (0 and 1).
# Never one worker spanning both GPUs, never a silent 2 -> 1 degrade.
if len(GPUS) == 1:
    log(
        "gpus         : exactly one physical GPU attached — allowing ONE worker pinned to cuda:0 "
        "(permitted only because there is a single device)"
    )
    assigned = GPUS[:1]
else:
    assigned = GPUS[:2]
    if len(GPUS) > 2:
        log(f"gpus         : ignoring extra GPUs {[gpu['index'] for gpu in GPUS[2:]]} — using 0 and 1")

WORKER_PLAN = []
for slot, gpu in enumerate(assigned):
    cuda_device, port = WORKER_SPECS[slot]
    context = ctx_for(gpu["gib"])
    WORKER_PLAN.append(
        {
            "slot": slot,
            "cuda": cuda_device,
            "port": port,
            "ctx": context,
            "kv_gib": context * KV_BYTES_PER_TOKEN / 2**30,
        }
    )
    log(
        f"worker {slot}     : cuda:{cuda_device} port={port} ctx={context} "
        f"kv={WORKER_PLAN[-1]['kv_gib']:.2f} GiB"
    )

if len(WORKER_PLAN) != len(assigned):
    raise SystemExit("worker plan does not cover every assigned GPU")
if len(GPUS) >= 2 and len(WORKER_PLAN) != 2:
    raise SystemExit("two physical GPUs must produce two workers — refusing to degrade silently")
if any(worker["ctx"] > CTX_FULL for worker in WORKER_PLAN):
    raise SystemExit(f"context above the {CTX_FULL} cap")

In [ ]:
%%bash
# [shell 3/3] tailscale static binaries (userspace networking: no /dev/net/tun, no systemd here).
# Do NOT use install.sh on Colab - it wants systemd.
set -euo pipefail
ROOT="${CLEF_ROOT:-/content}"
[ -d "$ROOT" ] && [ -w "$ROOT" ] || ROOT="$HOME/.clef-colab"
mkdir -p "$ROOT/logs"

TS_DIR="${CLEF_TS_PREFIX:-/opt/tailscale}"
if ! mkdir -p "$TS_DIR" 2>/dev/null; then TS_DIR="$ROOT/tailscale"; mkdir -p "$TS_DIR"; fi
if [ ! -w "$TS_DIR" ]; then TS_DIR="$ROOT/tailscale"; mkdir -p "$TS_DIR"; fi

if [ -x "$TS_DIR/tailscaled" ] && [ -x "$TS_DIR/tailscale" ]; then
  echo "already installed in $TS_DIR"
else
  TMP="$(mktemp -d)"
  curl -fL --retry 3 --retry-delay 2 --progress-bar \
    -o "$TMP/ts.tgz" https://pkgs.tailscale.com/stable/tailscale_latest_amd64.tgz
  tar -xzf "$TMP/ts.tgz" -C "$TMP"
  install -m 0755 "$TMP"/tailscale_*/tailscale  "$TS_DIR/tailscale"
  install -m 0755 "$TMP"/tailscale_*/tailscaled "$TS_DIR/tailscaled"
  rm -rf "$TMP"
fi

mkdir -p /var/lib/tailscale 2>/dev/null || true
"$TS_DIR/tailscale" version
echo "TS_DIR=$TS_DIR"

In [ ]:
# [7/13] Plexus management client — alias + provider, every write read back
from __future__ import annotations

import json
import urllib.error
import urllib.request


class PlexusError(RuntimeError):
    pass


class PlexusClient:
    """The only Plexus state this notebook owns: one provider, one alias rung.

    Every setter is read-modify-write followed by a re-GET assertion. A 200 from PUT is not
    evidence that the value landed.
    """

    def __init__(
        self,
        base: str,
        admin_key: str,
        alias: str = ALIAS,
        provider: str = PROVIDER,
        model: str = MODEL,
    ) -> None:
        if not admin_key:
            raise PlexusError("PLEXUS_ADMIN_KEY is empty")
        self.base = base.rstrip("/")
        self.admin_key = admin_key
        self.alias = alias
        self.provider = provider
        self.model = model

    # ------------------------------------------------------------------ plumbing
    def _req(self, method: str, path: str, payload: dict | None = None, timeout: float = 60.0) -> dict:
        data = json.dumps(payload).encode() if payload is not None else None
        request = urllib.request.Request(f"{self.base}{path}", data=data, method=method)
        request.add_header("x-admin-key", self.admin_key)
        request.add_header("accept", "application/json")
        if data is not None:
            request.add_header("content-type", "application/json")
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                raw = response.read()
                return json.loads(raw) if raw else {}
        except urllib.error.HTTPError as exc:
            body = exc.read()[:800].decode("utf-8", "replace")
            raise PlexusError(f"{method} {path} -> {exc.code}: {body}") from None
        except urllib.error.URLError as exc:
            raise PlexusError(f"{method} {path} -> unreachable: {exc.reason}") from None
        except json.JSONDecodeError as exc:
            raise PlexusError(f"{method} {path} -> non-JSON body: {exc}") from None

    # ------------------------------------------------------------------ provider
    def get_provider(self) -> dict:
        return self._req("GET", f"/v0/management/providers/{self.provider}")

    def put_provider(self, config: dict) -> None:
        self._req("PUT", f"/v0/management/providers/{self.provider}", config)

    def provider_concurrency(self) -> int | None:
        value = self.get_provider().get("maxConcurrency")
        return int(value) if isinstance(value, int) else None

    def set_provider_concurrency(self, worker_count: int) -> int:
        """Set `maxConcurrency` to the live worker count, preserving every other provider field.

        `plexus_setup.py` pins a baseline of 2 because a one-time setup cannot know how many GPUs
        a run will get. A one-worker run advertising two slots queues the second call behind a
        131k-context decode, so the value is rewritten per run and re-GET-ed: a 200 from PUT is
        not evidence that the value landed.
        """
        wanted = max(1, int(worker_count))
        config = self.get_provider()
        if config.get("maxConcurrency") == wanted:
            return wanted
        config["maxConcurrency"] = wanted
        self.put_provider(config)
        stored = self.get_provider().get("maxConcurrency")
        if stored != wanted:
            raise PlexusError(f"maxConcurrency={wanted} did not stick (provider reports {stored!r})")
        return wanted

    def provider_timeout_ms(self) -> int | None:
        value = self.get_provider().get("timeoutMs")
        return int(value) if isinstance(value, int) else None

    def set_provider_timeout_ms(self, value: int) -> int:
        """Raise `timeoutMs` to `value` if the stored one is lower, then prove it on a fresh GET.

        `plexus_setup.py` is a one-time script; nothing guarantees the provider still carries the
        timeout it wrote (an older run, a manual edit, a re-created provider). 60 s silently
        truncates every real decode, so the value in force is read and repaired here.
        """
        config = self.get_provider()
        current = config.get("timeoutMs")
        if isinstance(current, int) and current >= int(value):
            return current
        config["timeoutMs"] = int(value)
        self.put_provider(config)
        stored = self.get_provider().get("timeoutMs")
        if stored != int(value):
            raise PlexusError(f"timeoutMs={value} did not stick (provider reports {stored!r})")
        return int(stored)

    def set_provider_base(self, ip: str, port: int) -> str:
        """Point the provider's `systemone` surface at http://<tailnet ip>:<port>/v1."""
        base_url = f"http://{ip}:{port}/v1"
        config = self.get_provider()
        urls = dict(config.get("api_base_url") or {})
        urls["systemone"] = base_url
        config["api_base_url"] = urls
        self.put_provider(config)
        stored = (self.get_provider().get("api_base_url") or {}).get("systemone")
        if stored != base_url:
            raise PlexusError(f"provider base URL did not stick: wanted {base_url!r}, got {stored!r}")
        return stored

    # --------------------------------------------------------------------- alias
    def get_alias(self) -> dict:
        return self._req("GET", f"/v0/management/aliases/{self.alias}")

    def put_alias(self, alias: dict) -> dict:
        body = {key: value for key, value in alias.items() if key != "slug"}  # slug is a read-only echo
        self._req("PUT", f"/v0/management/aliases/{self.alias}", body)
        return self.get_alias()

    def _local_target(self, alias: dict) -> dict:
        for group in alias.get("target_groups") or []:
            if group.get("name") != "LOCAL":
                continue
            for target in group.get("targets") or []:
                if target.get("provider") == self.provider and target.get("model") == self.model:
                    return target
        listing = [
            (group.get("name"), target.get("provider"), target.get("model"), target.get("enabled"))
            for group in alias.get("target_groups") or []
            for target in (group.get("targets") or [])
        ]
        raise PlexusError(
            f"{self.provider}:{self.model} is not a target in the LOCAL group — targets are {listing}"
        )

    def rung_enabled(self) -> bool:
        return bool(self._local_target(self.get_alias()).get("enabled"))

    def set_rung_enabled(self, enabled: bool) -> bool:
        """Flip this rung's `enabled` in the LOCAL group, then prove it on a fresh GET."""
        alias = self.get_alias()
        target = self._local_target(alias)
        if bool(target.get("enabled")) == bool(enabled):
            return bool(self._local_target(self.get_alias()).get("enabled"))
        target["enabled"] = bool(enabled)
        fresh = self.put_alias(alias)
        stored = bool(self._local_target(fresh).get("enabled"))
        if stored != bool(enabled):
            raise PlexusError(f"rung enabled={enabled} did not stick (alias reports {stored})")
        return stored

    def describe_rung(self) -> str:
        try:
            alias = self.get_alias()
            target = self._local_target(alias)
            return f"{self.alias} LOCAL {self.provider}:{self.model} enabled={bool(target.get('enabled'))}"
        except PlexusError as exc:
            return f"unknown ({exc})"

In [ ]:
# [8/13] llama-server workers + the round-robin proxy that fronts them
from __future__ import annotations

import hmac
import http.client
import json
import os
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

BEARER = SECRETS["CLEF_REMOTE_KEY"]


def http_json(port: int, path: str, timeout: float = 5.0, bearer: str | None = None) -> dict:
    request = urllib.request.Request(f"http://127.0.0.1:{port}{path}")
    if bearer:
        request.add_header("authorization", f"Bearer {bearer}")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        raw = response.read()
        return json.loads(raw) if raw else {}


class Worker:
    """One llama-server process pinned to one physical GPU, loopback only."""

    def __init__(
        self,
        slot: int,
        cuda: int,
        port: int,
        ctx: int,
        model_path,
        mmproj=None,
        bearer: str = BEARER,
    ) -> None:
        self.slot = slot
        self.cuda = cuda
        self.port = port
        self.ctx = ctx
        self.model = model_path
        self.mmproj = mmproj
        self.bearer = bearer
        self.log_path = LOG_DIR / f"llama-{slot}.log"
        self.proc = None
        self._logfh = None

    def command(self) -> list:
        cmd = [
            str(LLAMA_SERVER),
            "-m",
            str(self.model),
            "--host",
            "127.0.0.1",
            "--port",
            str(self.port),
            "-c",
            str(self.ctx),
            "-b",
            "2048",
            "-ub",
            "512",
            "-np",
            "1",
            "-ngl",
            "99",
            "--api-key",
            self.bearer,
        ]
        if self.mmproj:
            cmd += ["--mmproj", str(self.mmproj)]
        return cmd

    def start(self) -> None:
        env = dict(os.environ)
        env["CUDA_VISIBLE_DEVICES"] = str(self.cuda)
        env["LD_LIBRARY_PATH"] = LIB_PATH + (
            os.pathsep + env["LD_LIBRARY_PATH"] if env.get("LD_LIBRARY_PATH") else ""
        )
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self._logfh = open(self.log_path, "ab", buffering=0)
        cmd = self.command()
        printable = " ".join("***" if arg == self.bearer else arg for arg in cmd)
        log(f"worker {self.slot}     : CUDA_VISIBLE_DEVICES={self.cuda} {printable}")
        self.proc = subprocess.Popen(
            cmd,
            stdout=self._logfh,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )
        log(f"worker {self.slot}     : pid={self.proc.pid} log={self.log_path}")

    def alive(self) -> bool:
        return self.proc is not None and self.proc.poll() is None

    def healthy(self, timeout: float = 3.0) -> bool:
        if not self.alive():
            return False
        try:
            body = http_json(self.port, "/health", timeout=timeout, bearer=self.bearer)
        except Exception:
            return False
        return str(body.get("status", "")).lower() == "ok"

    def props(self) -> dict:
        return http_json(self.port, "/props", timeout=30.0, bearer=self.bearer)

    def wait_ready(self, timeout: float = READY_TIMEOUT) -> dict:
        deadline = time.time() + timeout
        while time.time() < deadline:
            if self.proc is not None and self.proc.poll() is not None:
                raise RuntimeError(
                    f"worker {self.slot} exited rc={self.proc.returncode} while loading "
                    f"(see {self.log_path})\n{self.tail()}"
                )
            if self.healthy():
                return self.props()
            time.sleep(2.0)
        raise RuntimeError(f"worker {self.slot} never became healthy in {timeout:.0f}s\n{self.tail()}")

    def tail(self, lines: int = 40) -> str:
        try:
            with open(self.log_path, "r", errors="replace") as handle:
                return "".join(handle.readlines()[-lines:])
        except FileNotFoundError:
            return "(no log)"

    def stop(self, grace: float = 15.0) -> None:
        proc = self.proc
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    proc.terminate()
                except Exception:
                    pass
            try:
                proc.wait(timeout=grace)
            except subprocess.TimeoutExpired:
                log(f"worker {self.slot}     : ignored SIGTERM for {grace:.0f}s — SIGKILL")
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                except (ProcessLookupError, PermissionError, OSError):
                    try:
                        proc.kill()
                    except Exception:
                        pass
                try:
                    proc.wait(timeout=10)
                except Exception:
                    pass
        self.proc = None
        if self._logfh is not None:
            try:
                self._logfh.close()
            except Exception:
                pass
            self._logfh = None


class RoundRobinProxy:
    """Loopback front door for the workers.

    `Authorization: Bearer <CLEF_REMOTE_KEY>` is required on every inference route (401
    otherwise). POST /v1/systemone is spread round-robin across live workers and retried on the
    next worker after a connection failure. GET /health is deliberately unauthenticated: it is
    liveness only, and it is what the operator polls from the tailnet.
    """

    def __init__(self, workers, port: int, bearer: str = BEARER, upstream_timeout: float = UPSTREAM_TIMEOUT) -> None:
        self.workers = list(workers)
        self.port = port
        self.bearer = bearer
        self.upstream_timeout = upstream_timeout
        self.requests = 0
        self.served = 0
        self.failures = 0
        self._rr = 0
        self._lock = threading.Lock()
        self._server = None
        self._thread = None

    # ---------------------------------------------------------------- lifecycle
    def start(self) -> None:
        proxy = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = "clef-rr-proxy"

            def log_message(self, fmt: str, *args) -> None:  # noqa: A003 - stdlib signature
                log(f"proxy        : {self.address_string()} {fmt % args}")

            # --- helpers
            def _send(self, code: int, body: bytes, ctype: str = "application/json") -> None:
                self.send_response(code)
                self.send_header("content-type", ctype)
                self.send_header("content-length", str(len(body)))
                self.send_header("connection", "close")
                self.end_headers()
                self.close_connection = True
                try:
                    self.wfile.write(body)
                except (BrokenPipeError, ConnectionResetError):
                    pass

            def _json(self, code: int, payload: dict) -> None:
                self._send(code, json.dumps(payload).encode())

            def _unauthorized(self) -> None:
                self._json(401, {"error": "unauthorized", "detail": "Authorization: Bearer <key> required"})

            def _authorized(self) -> bool:
                header = self.headers.get("authorization", "") or ""
                if not header.lower().startswith("bearer "):
                    return False
                return hmac.compare_digest(header[7:].strip(), proxy.bearer)

            def _read_body(self) -> bytes | None:
                length = self.headers.get("content-length")
                if length is None:
                    if (self.headers.get("transfer-encoding") or "").lower() == "chunked":
                        self._json(411, {"error": "chunked bodies are not supported; send content-length"})
                        return None
                    return b""
                try:
                    return self.rfile.read(int(length))
                except (ValueError, OSError):
                    self._json(400, {"error": "unreadable content-length"})
                    return None

            def _forward(self, method: str, path: str, body: bytes | None) -> None:
                headers = {
                    "authorization": f"Bearer {proxy.bearer}",
                    "content-type": "application/json",
                    "accept": "application/json",
                    "connection": "close",
                }
                last_error = None
                for worker in proxy.order():
                    try:
                        conn = http.client.HTTPConnection(
                            "127.0.0.1", worker.port, timeout=proxy.upstream_timeout
                        )
                        conn.request(method, path, body=body, headers=headers)
                        response = conn.getresponse()
                        data = response.read()
                        status = response.status
                        ctype = response.getheader("content-type") or "application/json"
                        conn.close()
                        with proxy._lock:
                            proxy.served += 1
                        self._send(status, data, ctype)
                        return
                    except socket.timeout as exc:
                        last_error = exc
                        with proxy._lock:
                            proxy.failures += 1
                        log(f"proxy        : worker {worker.slot}:{worker.port} timed out; trying next")
                    except (ConnectionError, OSError, http.client.HTTPException) as exc:
                        last_error = exc
                        with proxy._lock:
                            proxy.failures += 1
                        log(f"proxy        : worker {worker.slot}:{worker.port} failed ({exc!r}); trying next")
                self._json(502, {"error": "no worker answered", "detail": repr(last_error)})

            # --- routes
            def do_GET(self) -> None:  # noqa: N802 - stdlib signature
                path = self.path.split("?", 1)[0]
                if path == "/health":
                    self._json(
                        200,
                        {
                            "status": "ok",
                            "workers": [
                                {
                                    "slot": worker.slot,
                                    "port": worker.port,
                                    "cuda": worker.cuda,
                                    "alive": worker.alive(),
                                    "healthy": worker.healthy(),
                                }
                                for worker in proxy.workers
                            ],
                            "requests": proxy.requests,
                            "served": proxy.served,
                            "failures": proxy.failures,
                        },
                    )
                    return
                if path in ("/v1/models", "/models"):
                    if not self._authorized():
                        self._unauthorized()
                        return
                    with proxy._lock:
                        proxy.requests += 1
                    self._forward("GET", "/v1/models", None)
                    return
                self._json(404, {"error": "not found", "path": path})

            def do_POST(self) -> None:  # noqa: N802 - stdlib signature
                path = self.path.split("?", 1)[0]
                if path not in ("/v1/systemone", "/systemone"):
                    self._json(404, {"error": "not found", "path": path, "hint": "POST /v1/systemone"})
                    return
                if not self._authorized():
                    self._unauthorized()
                    return
                body = self._read_body()
                if body is None:
                    return
                with proxy._lock:
                    proxy.requests += 1
                self._forward("POST", "/v1/systemone", body)

        self._server = ThreadingHTTPServer(("127.0.0.1", self.port), Handler)
        self._server.daemon_threads = True
        self._thread = threading.Thread(target=self._server.serve_forever, name="rr-proxy", daemon=True)
        self._thread.start()

    def order(self) -> list:
        """Live workers, round-robin starting point; falls back to every worker if none are up."""
        with self._lock:
            live = [worker for worker in self.workers if worker.alive()] or list(self.workers)
            start = self._rr % len(live)
            self._rr += 1
            return live[start:] + live[:start]

    def stop(self) -> None:
        server, self._server = self._server, None
        if server is None:
            return
        try:
            server.shutdown()
        except Exception:
            pass
        try:
            server.server_close()
        except Exception:
            pass
        if self._thread is not None:
            try:
                self._thread.join(timeout=5)
            except Exception:
                pass
            self._thread = None

In [ ]:
# [9/13] tailscale: userspace daemon, ephemeral join, TCP serve
from __future__ import annotations

import json
import os
import re
import signal
import socket
import subprocess
import time
import urllib.error
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

NODE_STALE_SECONDS = 300


class Tailscale:
    """Userspace-mode tailscaled + CLI, on a private state file and socket."""

    def __init__(
        self,
        ts_dir=TS_DIR,
        sock=TS_SOCK,
        state=TS_STATE,
        hostname: str = TS_HOSTNAME,
        api_key: str = "",
    ) -> None:
        self.dir = Path(ts_dir)
        self.sock = Path(sock)
        self.state = Path(state)
        self.hostname = hostname
        self.api_key = api_key or SECRETS["TAILSCALE_API_KEY"]
        self.proc = None
        self._logfh = None
        self.device_id = None
        self.auth_key_id = None

    @property
    def cli(self) -> str:
        return str(self.dir / "tailscale")

    @property
    def daemon(self) -> str:
        return str(self.dir / "tailscaled")

    @staticmethod
    def _redacted(args: list) -> str:
        """Command line safe to log: an authkey never survives into a message."""
        return " ".join(
            "--authkey=***" if str(arg).startswith("--authkey=") else str(arg) for arg in args
        )

    def _run(self, args: list, timeout: float = 60.0, check: bool = True) -> subprocess.CompletedProcess:
        try:
            proc = subprocess.run(
                [self.cli, f"--socket={self.sock}", *args],
                capture_output=True,
                text=True,
                timeout=timeout,
            )
        except subprocess.TimeoutExpired:
            # TimeoutExpired.__str__ embeds the whole argv, i.e. --authkey=<secret>
            raise RuntimeError(
                f"tailscale {self._redacted(args)} timed out after {timeout:.0f}s"
            ) from None
        if check and proc.returncode != 0:
            detail = (proc.stderr.strip() or proc.stdout.strip())[:600]
            raise RuntimeError(f"tailscale {self._redacted(args)} rc={proc.returncode}: {detail}")
        return proc

    # ------------------------------------------------------------- tailscale api
    def _api(self, method: str, path: str, payload: dict | None = None, timeout: float = 45.0, allowed: tuple = (200,)) -> tuple:
        """Call the Tailscale REST API. Never logs the API token or an auth key value."""
        data = json.dumps(payload).encode() if payload is not None else None
        request = urllib.request.Request(f"{TS_API_BASE}{path}", data=data, method=method)
        request.add_header("authorization", f"Bearer {self.api_key}")
        request.add_header("accept", "application/json")
        if data is not None:
            request.add_header("content-type", "application/json")
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                status = response.status
                raw = response.read()
        except urllib.error.HTTPError as exc:
            body = exc.read()[:600].decode("utf-8", "replace")
            raise RuntimeError(f"tailscale api {method} {path} -> {exc.code}: {body}") from None
        except urllib.error.URLError as exc:
            raise RuntimeError(f"tailscale api {method} {path} -> unreachable: {exc.reason}") from None
        if status not in allowed:
            raise RuntimeError(f"tailscale api {method} {path} -> unexpected status {status}")
        if not raw:
            return status, {}
        try:
            return status, json.loads(raw)
        except json.JSONDecodeError:
            return status, {}

    def devices(self) -> list:
        _status, body = self._api("GET", "/tailnet/-/devices")
        devices = body.get("devices") if isinstance(body, dict) else None
        if not isinstance(devices, list):
            raise RuntimeError(f"Tailscale device list had an unexpected shape: {body!r:.400}")
        return devices

    def _is_notebook_device(self, device: dict) -> bool:
        """Match exactly `clef-llama-remote` or `clef-llama-remote-<digits>`."""
        names = set()
        hostname = (device.get("hostname") or "").strip().rstrip(".")
        if hostname:
            names.add(hostname)
        full = (device.get("name") or "").strip().rstrip(".")
        if full:
            names.add(full.split(".")[0])
        pattern = re.compile(re.escape(self.hostname) + r"-\d+\Z")
        return any(name == self.hostname or pattern.match(name) for name in names)

    def cleanup_duplicate_nodes(self) -> list:
        """Delete stale notebook nodes before joining, but never a freshly-seen one.

        A matching hostname is not proof of staleness: another live notebook uses the same
        name. Refuse instead of deleting a node that answered within NODE_STALE_SECONDS.
        """
        stale = [device for device in self.devices() if self._is_notebook_device(device)]
        fresh = []
        for device in stale:
            last_seen = device.get("lastSeen")
            if not last_seen:
                continue
            try:
                seen = datetime.fromisoformat(last_seen.replace("Z", "+00:00"))
                age = (datetime.now(timezone.utc) - seen).total_seconds()
            except (TypeError, ValueError):
                continue
            if age < NODE_STALE_SECONDS:
                fresh.append((device.get("hostname") or device.get("name"), last_seen, int(age)))
        if fresh:
            raise RuntimeError(
                "refusing to delete recently-seen notebook node(s): "
                f"{fresh}. Another live notebook may own this name; stop it or delete the node "
                "manually, then re-run. Fresh threshold is "
                f"{NODE_STALE_SECONDS} s."
            )
        removed = []
        for device in stale:
            device_id = device.get("id")
            if not device_id:
                continue
            self._api("DELETE", f"/device/{device_id}", allowed=(200, 204, 404))
            removed.append({"hostname": device.get("hostname") or device.get("name"), "lastSeen": device.get("lastSeen")})
            time.sleep(0.2)
        remaining = [device for device in self.devices() if self._is_notebook_device(device)]
        if remaining:
            listing = [(device.get("hostname"), device.get("lastSeen")) for device in remaining]
            raise RuntimeError(f"duplicate Tailscale nodes survived deletion: {listing}")
        if removed:
            log(f"tailscale    : deleted {len(removed)} duplicate node(s): {removed}")
        else:
            log("tailscale    : no duplicate clef-llama-remote nodes")
        return removed

    def mint_auth_key(self, expiry_seconds: int = TS_AUTHKEY_EXPIRY_SECONDS) -> str:
        """Mint a single-use, ephemeral, preauthorised key for this join, then return its value."""
        description = f"clef-llama-remote-colab-v2-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}"
        _status, body = self._api(
            "POST",
            "/tailnet/-/keys",
            {
                "capabilities": {"devices": {"create": {"reusable": False, "ephemeral": True, "preauthorized": True, "tags": [TS_TAG]}}},
                "expirySeconds": int(expiry_seconds),
                "description": description,
            },
        )
        key = body.get("key") if isinstance(body, dict) else ""
        self.auth_key_id = body.get("id") if isinstance(body, dict) else None
        if not isinstance(key, str) or not key.startswith("tskey-auth-"):
            raise RuntimeError("Tailscale did not return an auth key value for this join")
        log(f"tailscale    : minted single-use ephemeral auth key (id={self.auth_key_id})")
        return key

    def revoke_auth_key(self) -> bool:
        """Best-effort delete of the auth key minted for this run."""
        key_id, self.auth_key_id = self.auth_key_id, None
        if not key_id:
            return False
        try:
            self._api("DELETE", f"/tailnet/-/keys/{key_id}", allowed=(200, 204, 404))
            log("tailscale    : per-run auth key revoked")
            return True
        except Exception as exc:
            log(f"tailscale    : auth-key revoke failed ({exc})")
            return False

    def own_device_id(self) -> str | None:
        """This run's Tailscale device id, matched by node id or tailnet addresses."""
        state = self.status()
        self_node = state.get("Self") or {}
        node_id = self_node.get("ID")
        addresses = set(self_node.get("TailscaleIPs") or [])
        for device in self.devices():
            if node_id and device.get("nodeId") == node_id:
                return str(device.get("id") or "") or None
            if addresses and addresses & set(device.get("addresses") or []):
                return str(device.get("id") or "") or None
        return None

    def delete_device(self, device_id: str | None) -> bool:
        """Delete a device by id. 404 means it is already gone and is treated as success."""
        if not device_id:
            return False
        try:
            self._api("DELETE", f"/device/{device_id}", allowed=(200, 204, 404))
            return True
        except Exception as exc:
            log(f"tailscale    : node delete failed ({exc})")
            return False

    # ------------------------------------------------------------------ daemon
    def start_daemon(self, timeout: float = 90.0) -> None:
        for path in (self.daemon, self.cli):
            if not Path(path).exists():
                raise SystemExit(f"{path} missing — run the tailscale cell ([shell 3/3]) first")
        if self.proc is not None and self.proc.poll() is None:
            log("tailscaled   : already running")
            return
        self.sock.parent.mkdir(parents=True, exist_ok=True)
        self.state.parent.mkdir(parents=True, exist_ok=True)
        log_path = LOG_DIR / "tailscaled.log"
        self._logfh = open(log_path, "ab", buffering=0)
        cmd = [
            self.daemon,
            "--tun=userspace-networking",
            "--socks5-server=localhost:1055",
            "--outbound-http-proxy-listen=localhost:1055",
            f"--state={self.state}",
            f"--socket={self.sock}",
        ]
        log(f"tailscaled   : {' '.join(cmd)}")
        self.proc = subprocess.Popen(
            cmd, stdout=self._logfh, stderr=subprocess.STDOUT, start_new_session=True
        )
        deadline = time.time() + timeout
        while time.time() < deadline:
            if self.proc.poll() is not None:
                raise RuntimeError(f"tailscaled exited rc={self.proc.returncode}; see {log_path}")
            if self._run(["status", "--json"], timeout=20.0, check=False).stdout.strip().startswith("{"):
                log(f"tailscaled   : socket answering ({self.sock})")
                return
            time.sleep(1.0)
        raise RuntimeError(f"tailscaled socket did not answer within {timeout:.0f}s; see {log_path}")

    # -------------------------------------------------------------------- state
    def status(self) -> dict:
        proc = self._run(["status", "--json"], timeout=30.0, check=False)
        text = proc.stdout.strip()
        if not text.startswith("{"):
            return {}
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {}

    def backend_state(self) -> str:
        return str(self.status().get("BackendState", "unknown"))

    # --------------------------------------------------------------------- join
    def up(self, authkey: str, timeout: float = 180.0) -> dict:
        proc = self._run(
            [
                "up",
                f"--authkey={authkey}",
                f"--hostname={self.hostname}",
                "--accept-dns=false",
                "--ssh=false",
            ],
            timeout=timeout,
            check=False,
        )
        if proc.returncode != 0:
            detail = (proc.stderr.strip() or proc.stdout.strip())[:600]
            raise RuntimeError(f"tailscale up failed rc={proc.returncode}: {detail}")
        return self.wait_running()

    def wait_running(self, timeout: float = 120.0) -> dict:
        deadline = time.time() + timeout
        state = {}
        while time.time() < deadline:
            state = self.status()
            if str(state.get("BackendState")) == "Running":
                return state
            time.sleep(2.0)
        raise RuntimeError(
            f"tailnet never reached Running (BackendState={state.get('BackendState')!r}); "
            "an expired or already-consumed authkey is the usual cause"
        )

    def ipv4(self) -> str:
        proc = self._run(["ip", "-4"], timeout=30.0, check=False)
        lines = [line.strip() for line in proc.stdout.splitlines() if line.strip()]
        ip = lines[0] if lines else ""
        if not ip.startswith("100."):
            raise RuntimeError(f"expected a 100.x tailnet IPv4, got {ip!r}")
        return ip

    def dnsname(self) -> str:
        return str((self.status().get("Self") or {}).get("DNSName", "")).rstrip(".")

    # -------------------------------------------------------------------- serve
    def serve_tcp(self, port: int) -> None:
        """Publish tcp/<port> on the tailnet, forwarding to the same loopback port."""
        self._run(["serve", "--bg", f"--tcp={port}", f"tcp://127.0.0.1:{port}"], timeout=90.0)

    def serve_reset(self) -> None:
        self._run(["serve", "reset"], timeout=60.0, check=False)

    # -------------------------------------------------------------------- probe
    def probe(self, ip: str, port: int, path: str = "/health", timeout: float = PROBE_TIMEOUT) -> tuple:
        """GET http://<ip>:<port><path> through tailscaled's userspace SOCKS5 proxy, from here.

        Colab has no /dev/net/tun, so a plain socket to 100.x cannot leave this netns; the
        daemon's outbound proxy is the only local route onto the tailnet, and a request sent
        through it leaves as a tailnet packet and comes back in through `tailscale serve`. That
        makes it a check that the published port answers, not merely that serve is configured.
        Returns (ok, detail); never raises.
        """

        def read_exact(sock, count: int) -> bytes:
            buf = b""
            while len(buf) < count:
                block = sock.recv(count - len(buf))
                if not block:
                    raise OSError("SOCKS5 proxy closed mid-handshake")
                buf += block
            return buf

        try:
            sock = socket.create_connection(("127.0.0.1", 1055), timeout=timeout)
        except OSError as exc:
            return False, f"tailscaled outbound proxy 127.0.0.1:1055 unreachable ({exc})"
        try:
            sock.settimeout(timeout)
            sock.sendall(b"\x05\x01\x00")
            if read_exact(sock, 2) != b"\x05\x00":
                return False, "SOCKS5 proxy rejected the no-auth greeting"
            host = ip.encode()
            sock.sendall(b"\x05\x01\x00\x03" + bytes([len(host)]) + host + port.to_bytes(2, "big"))
            head = read_exact(sock, 4)
            if head[1] != 0:
                return False, f"SOCKS5 CONNECT {ip}:{port} refused (rep={head[1]})"
            if head[3] == 1:
                read_exact(sock, 6)
            elif head[3] == 4:
                read_exact(sock, 18)
            elif head[3] == 3:
                read_exact(sock, read_exact(sock, 1)[0] + 2)
            else:
                return False, f"SOCKS5 CONNECT {ip}:{port} returned atyp={head[3]}"
            sock.sendall(
                f"GET {path} HTTP/1.0\r\nhost: {ip}:{port}\r\nconnection: close\r\n\r\n".encode()
            )
            chunks = []
            while True:
                block = sock.recv(4096)
                if not block:
                    break
                chunks.append(block)
            raw = b"".join(chunks)
            status = raw.split(b"\r\n", 1)[0].decode("latin-1", "replace").strip()
            if " 200" not in status:
                return False, f"{ip}:{port}{path} -> {status or 'empty response'}"
            return True, f"{ip}:{port}{path} -> {status}"
        except (OSError, ValueError) as exc:
            return False, f"{ip}:{port}{path} failed: {exc}"
        finally:
            try:
                sock.close()
            except Exception:
                pass

    def down(self) -> None:
        self._run(["down"], timeout=90.0, check=False)

    def stop_daemon(self, grace: float = 10.0) -> None:
        proc, self.proc = self.proc, None
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    proc.terminate()
                except Exception:
                    pass
            try:
                proc.wait(timeout=grace)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                except (ProcessLookupError, PermissionError, OSError):
                    try:
                        proc.kill()
                    except Exception:
                        pass
        if self._logfh is not None:
            try:
                self._logfh.close()
            except Exception:
                pass
            self._logfh = None

In [ ]:
# [10/13] Runtime: bring-up order, teardown order, heartbeat
from __future__ import annotations

import atexit
import json
import os
import signal
import threading
import time
import urllib.error
import urllib.request


class SelfTestFailure(RuntimeError):
    pass


class Runtime:
    """Owns the whole ephemeral node: workers, proxy, tailnet, Plexus registration.

    Bring-up:  workers -> GPU residency proof -> tailscale -> proxy -> tailnet reachability
               -> provider base URL/concurrency/timeout -> rung enabled.
    Teardown:  rung disabled FIRST (so Plexus stops routing), then the configured concurrency,
               then workers, then the proxy, then the tailnet.
    """

    def __init__(self, client: PlexusClient | None = None, ts: Tailscale | None = None) -> None:
        self.client = client or PlexusClient(PLEXUS_BASE, SECRETS["PLEXUS_ADMIN_KEY"])
        self.ts = ts or Tailscale()
        self.workers: list = []
        self.proxy = None
        self.serve_port = None
        self.ip = None
        self.dnsname = None
        self.device_id = None
        self.base_url = None
        self._stop = threading.Event()
        self._done = threading.Event()
        self._guard = threading.Lock()
        self._stopped = False
        self._rung_on = False
        self._armed = False
        self._rung_warned = False
        self._heartbeat = None
        self.gpu_mib = {}
        self.concurrency = None
        self.provider_timeout_ms = None
        self._concurrency_baseline = None

    # ------------------------------------------------------------------ bring-up
    def start(self) -> "Runtime":
        with self._guard:
            if self._stopped:
                raise RuntimeError("this Runtime was already stopped — construct a new one")
            if self.workers or self.proxy is not None:
                raise RuntimeError("Runtime.start() was already called in this kernel")
        self._arm_callbacks()
        log(
            "bring-up     : workers -> gpu proof -> tailscale -> proxy -> reachability "
            "-> provider -> rung"
        )
        try:
            for stage in (
                self._start_workers,
                self._prove_gpu,
                self._start_tailnet,
                self._start_proxy,
                self._prove_reachable,
                self._register_provider,
            ):
                if stage in (self._start_tailnet, self._start_proxy, self._register_provider):
                    # These phases create external state (tailnet, Serve, provider config).
                    # Serialize them with stop() so teardown cannot miss late side effects.
                    with self._guard:
                        self._ensure_running()
                        stage()
                else:
                    self._ensure_running()
                    stage()
            # Serialize the final enable against stop(): if teardown wins the guard, no later
            # stage may turn the rung back on after the workers and tailnet are being closed.
            with self._guard:
                self._ensure_running()
                self._enable_rung()
            self._ensure_running()
            self._self_test()
            self._ensure_running()
        except BaseException:
            log("bring-up     : failed — tearing down (rung disabled first)")
            self.stop(reason="bring-up failed")
            raise
        self._heartbeat = threading.Thread(target=self._watch, name="heartbeat", daemon=True)
        self._heartbeat.start()
        self.status()
        log("bring-up     : ready")
        return self

    def _ensure_running(self) -> None:
        if self._stop.is_set() or self._stopped:
            raise RuntimeError("teardown requested during bring-up — refusing to continue")

    def _start_workers(self) -> None:
        for spec in WORKER_PLAN:
            worker = Worker(
                slot=spec["slot"],
                cuda=spec["cuda"],
                port=spec["port"],
                ctx=spec["ctx"],
                model_path=GGUF,
                mmproj=MMPROJ,
            )
            # Spawn inside the guard: a teardown arriving mid-bring-up must never sweep the worker
            # list between append() and Popen(), which would leave a llama-server holding ~10 GB of
            # VRAM on a node whose rung is already disabled.
            with self._guard:
                if self._stopped:
                    raise RuntimeError("teardown requested during bring-up — not starting another worker")
                worker.start()
                self.workers.append(worker)
        for worker in self.workers:
            props = worker.wait_ready()
            settings = props.get("default_generation_settings") or {}
            reported = settings.get("n_ctx")
            log(f"worker {worker.slot}     : ready n_ctx={reported} (requested {worker.ctx})")
            if reported is not None and int(reported) != worker.ctx:
                log(
                    f"WARNING: worker {worker.slot} reports n_ctx={reported} but {worker.ctx} was "
                    "requested — the server clamped it (VRAM or --parallel)"
                )

    def _prove_gpu(self) -> None:
        """Refuse to continue unless every worker is provably holding GPU memory.

        `-ngl 99` is a request, not a result: a CPU build, a stale binary or a CUDA init that
        fell back all keep the weights on the host and still answer /health. Enabling the rung on
        such a node burns the provider timeout on every call, so this gate is what keeps it off.

        Evidence order: nvidia-smi compute-apps memory per worker pid (primary, no log parsing),
        then an "offloaded N/M layers" line from the worker log (secondary, only when the driver
        declines to enumerate compute processes).
        """
        deadline = time.time() + 30.0
        held = {}
        pending = []
        while True:
            held = gpu_compute_memory()
            pending = [
                worker.slot
                for worker in self.workers
                if held.get(worker.proc.pid if worker.proc is not None else -1, 0) < GPU_MIN_USED_MIB
            ]
            if not held or not pending or time.time() >= deadline:
                break  # no compute-apps view at all: go straight to the log fallback
            time.sleep(5.0)  # nvidia-smi lags context creation by a moment
        if held:
            if pending:
                raise RuntimeError(
                    f"worker(s) {pending} hold less than {GPU_MIN_USED_MIB} MiB on a GPU "
                    f"(nvidia-smi compute apps: {held!r}): the weights are not on the card, so the "
                    "rung stays disabled. Check that the CUDA release is the one in use, that "
                    "LD_LIBRARY_PATH carries the cudart directory, and read the worker log."
                )
            self.gpu_mib = {
                worker.slot: held.get(worker.proc.pid, 0)
                for worker in self.workers
                if worker.proc is not None
            }
            evidence = ", ".join(f"slot {slot}={mib} MiB" for slot, mib in sorted(self.gpu_mib.items()))
            log(f"gpu proof    : resident GPU memory per worker: {evidence}")
            return
        # nvidia-smi would not enumerate compute processes: fall back to the load log.
        for worker in self.workers:
            layers = gpu_layers_in_log(worker.log_path)
            if layers is None or layers[0] <= 0:
                raise RuntimeError(
                    f"cannot prove worker {worker.slot} put layers on the GPU: nvidia-smi lists no "
                    f"compute processes and {worker.log_path} has no offload line ({layers!r}). "
                    "Refusing to enable the rung on an unproven load."
                )
            log(f"gpu proof    : worker {worker.slot} log reports {layers[0]}/{layers[1]} layers offloaded")

    def _prove_reachable(self) -> None:
        """Prove the published port answers over the tailnet before Plexus is told about it."""
        ok, detail = self.ts.probe(self.ip, self.serve_port)
        if ok:
            log(f"tailnet      : reachability probe OK ({detail})")
            return
        log(f"tailnet      : reachability probe FAILED ({detail})")
        if os.environ.get("CLEF_ALLOW_UNREACHABLE", "").strip().lower() in ("1", "true", "yes", "on"):
            log("tailnet      : CLEF_ALLOW_UNREACHABLE set — registering without a reachability proof")
            return
        raise RuntimeError(
            f"tcp/{self.serve_port} does not answer over the tailnet ({detail}). Plexus would be "
            "pointed at a rung it cannot reach, so bring-up stops here. Check `tailscale serve "
            "status`, BackendState and the tag:endpoints ACL; set CLEF_ALLOW_UNREACHABLE=1 to skip "
            "this gate deliberately."
        )

    def _start_tailnet(self) -> None:
        # Cleanup and key minting use the API token, so they run before the daemon exists.
        self.ts.cleanup_duplicate_nodes()
        self.ts.start_daemon()
        authkey = self.ts.mint_auth_key()
        try:
            self.ts.up(authkey)
        finally:
            self.ts.revoke_auth_key()
        self.ip = self.ts.ipv4()
        self.dnsname = self.ts.dnsname()
        self.device_id = self.ts.own_device_id()
        log(f"tailnet      : {self.ip}  {self.dnsname or '(no DNSName)'}")
        if self.device_id:
            log(f"tailnet      : device id {self.device_id} will be deleted on teardown")
        else:
            log(
                "WARNING: could not resolve this node's Tailscale API device id — teardown cannot "
                "delete it; the next v2 run's duplicate cleanup will."
            )
        if EXPECTED_DNSNAME and self.dnsname and self.dnsname != EXPECTED_DNSNAME:
            bar = "=" * 78
            log(bar)
            log(f"WARNING: joined as {self.dnsname!r}, expected {EXPECTED_DNSNAME!r}")
            log("  Cleanup should have removed every stale node. Another live notebook is the most")
            log("  likely cause; stop it, then re-run this cell. Routing uses the IP, not the name.")
            log(bar)

    def _start_proxy(self) -> None:
        last_error = None
        for port in SERVE_PORT_LADDER:
            proxy = RoundRobinProxy(self.workers, port)
            try:
                proxy.start()
            except OSError as exc:
                last_error = exc
                log(f"proxy        : 127.0.0.1:{port} unavailable ({exc}); next candidate")
                continue
            try:
                self.ts.serve_tcp(port)
            except Exception as exc:
                last_error = exc
                log(f"tailscale    : serve --tcp={port} failed ({exc}); next candidate")
                proxy.stop()
                continue
            self.proxy = proxy
            self.serve_port = port
            log(f"proxy        : 127.0.0.1:{port} round-robin over {[w.port for w in self.workers]}")
            log(f"tailscale    : serve tcp {port} -> 127.0.0.1:{port}")
            return
        raise RuntimeError(f"no usable serve port in {list(SERVE_PORT_LADDER)}: {last_error!r}")

    def _register_provider(self) -> None:
        config = self.client.get_provider()
        entry = (config.get("models") or {}).get(MODEL) or {}
        if entry.get("type") != "decisions":
            raise RuntimeError(f"provider model {MODEL} is {entry.get('type')!r}, expected 'decisions'")
        if "systemone" not in (entry.get("access_via") or []):
            raise RuntimeError(f"provider model {MODEL} access_via={entry.get('access_via')!r}, expected systemone")
        baseline = config.get("maxConcurrency")
        self._concurrency_baseline = baseline if isinstance(baseline, int) else None
        self.base_url = self.client.set_provider_base(self.ip, self.serve_port)
        log(f"provider     : {PROVIDER}.api_base_url.systemone = {self.base_url} (re-GET verified)")
        self.concurrency = self.client.set_provider_concurrency(len(WORKER_PLAN))
        log(
            f"provider     : maxConcurrency={self.concurrency} for {len(WORKER_PLAN)} worker(s) "
            f"(configured baseline {self._concurrency_baseline}, re-GET verified)"
        )
        self.provider_timeout_ms = self.client.provider_timeout_ms()
        if self.provider_timeout_ms is None or self.provider_timeout_ms < MIN_PROVIDER_TIMEOUT_MS:
            log(
                f"provider     : timeoutMs={self.provider_timeout_ms} is under the "
                f"{MIN_PROVIDER_TIMEOUT_MS} ms floor — raising it"
            )
            self.provider_timeout_ms = self.client.set_provider_timeout_ms(PROVIDER_TIMEOUT_MS)
        if self.provider_timeout_ms < UPSTREAM_TIMEOUT * 1000:
            raise RuntimeError(
                f"provider timeoutMs={self.provider_timeout_ms} does not exceed the proxy's "
                f"{UPSTREAM_TIMEOUT:.0f} s upstream timeout: Plexus would abandon requests the proxy "
                "is still serving. Fix the provider or lower UPSTREAM_TIMEOUT."
            )
        log(f"provider     : timeoutMs={self.provider_timeout_ms} (re-GET verified)")

    def _enable_rung(self) -> None:
        self.client.set_rung_enabled(True)
        self._rung_on = True
        log(f"rung         : {self.client.describe_rung()} (re-GET verified)")

    # ----------------------------------------------------------------- self-test
    def _self_test(self) -> None:
        api_key = SECRETS["PLEXUS_API_KEY"]
        questions = {
            "fruit": {
                "type": "choice",
                "instructions": "Which of these is a fruit?",
                "criteria": {"apple": "a pomaceous fruit", "potato": "a root vegetable"},
            },
            "sure": {
                "type": "noul",
                "instructions": "How sure are you of that answer?",
            },
        }
        payload = {
            "model": SELF_TEST_MODEL,
            "state": "colab bring-up self-test (direct provider route)",
            "questions": questions,
        }
        url = f"{PLEXUS_BASE}/v1/systemone"
        served_before = self.proxy.served if self.proxy is not None else 0
        request = urllib.request.Request(url, data=json.dumps(payload).encode(), method="POST")
        request.add_header("authorization", f"Bearer {api_key}")
        request.add_header("content-type", "application/json")
        request.add_header("accept", "application/json")
        try:
            with urllib.request.urlopen(request, timeout=SELF_TEST_TIMEOUT) as response:
                raw = response.read()
                headers = dict(response.headers)
        except urllib.error.HTTPError as exc:
            body = exc.read()[:1200].decode("utf-8", "replace")
            raise SelfTestFailure(f"POST {url} -> {exc.code}: {body}") from None
        except urllib.error.URLError as exc:
            raise SelfTestFailure(f"POST {url} -> unreachable: {exc.reason}") from None
        except Exception as exc:  # timeouts and resets arrive as assorted types
            raise SelfTestFailure(f"POST {url} -> {type(exc).__name__}: {exc}") from None
        try:
            body = json.loads(raw)
        except json.JSONDecodeError:
            raise SelfTestFailure(f"non-JSON response: {raw[:600]!r}") from None
        answers = body.get("answers")
        if not isinstance(answers, dict) or not answers:
            raise SelfTestFailure(f"empty `answers` in routing body: {json.dumps(body)[:1200]}")
        missing = [qid for qid in questions if qid not in answers]
        if missing:
            raise SelfTestFailure(f"answers missing ids {missing}: {json.dumps(body)[:1200]}")
        fruit = answers.get("fruit")
        if not isinstance(fruit, dict) or fruit.get("type") != "choice":
            raise SelfTestFailure(f"fruit answer is not a choice object: {fruit!r:.400}")
        if fruit.get("choice") not in questions["fruit"]["criteria"]:
            raise SelfTestFailure(
                f"fruit choice {fruit.get('choice')!r} is not one of the criteria keys: {fruit!r:.400}"
            )
        sure = answers.get("sure")
        if not isinstance(sure, dict) or sure.get("type") != "noul":
            raise SelfTestFailure(f"sure answer is not a noul object: {sure!r:.400}")
        value = sure.get("noul")
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not 0.0 <= float(value) <= 1.0:
            raise SelfTestFailure(f"sure.noul is not a number in [0,1]: {sure!r:.400}")
        routed = {key: body[key] for key in ("provider", "provider_name", "routed_via", "model") if key in body}
        hints = {key: value for key, value in headers.items() if key.lower().startswith("x-plexus")}
        log(f"self-test    : model={SELF_TEST_MODEL} answers={json.dumps(answers)[:400]}")
        log(f"self-test    : usage={json.dumps(body.get('usage'))} route={routed or hints or 'not disclosed'}")
        served_after = self.proxy.served if self.proxy is not None else 0
        if served_after <= served_before:
            raise SelfTestFailure(
                "Plexus returned answers, but this node's proxy served counter did not increase; "
                "the request routed to another rung. Check the LOCAL selector and enabled targets. "
                f"routing body: {json.dumps(body)[:600]}"
            )
        log(f"self-test    : served by this node (proxy served={served_after})")

    # ----------------------------------------------------------------- heartbeat
    def _watch(self) -> None:
        ts_misses = 0
        worker_misses = 0
        while not self._stop.wait(WATCH_INTERVAL):
            try:
                state = self.ts.backend_state()
            except Exception as exc:
                state = f"error:{exc!r}"
            if state == "Running":
                ts_misses = 0
            else:
                ts_misses += 1
                log(f"heartbeat    : BackendState={state!r} ({ts_misses}/2)")
                if ts_misses >= 2:
                    log("heartbeat    : tailnet is down — disabling the rung so Plexus stops routing here")
                    self.stop(reason=f"tailscale BackendState={state}")
                    return
            # One healthy worker still serves; every worker dead or unhealthy twice running does not.
            states = [(worker.slot, worker.alive(), worker.healthy()) for worker in self.workers]
            if states and not any(healthy for _, _, healthy in states):
                worker_misses += 1
                log(f"heartbeat    : no healthy worker {states} ({worker_misses}/2)")
                if worker_misses >= 2:
                    log("heartbeat    : disabling the rung before stopping the node")
                    self.stop(reason="every worker is down or unhealthy")
                    return
            else:
                worker_misses = 0
            try:
                enabled = self.client.rung_enabled()
            except Exception as exc:
                log(f"heartbeat    : rung check failed ({exc})")
                continue
            if enabled:
                self._rung_warned = False
            elif not self._rung_warned:
                self._rung_warned = True
                log(
                    f"WARNING: rung {PROVIDER}:{MODEL} was disabled externally — this node now serves "
                    "nothing. Re-run the enable step or the whole bring-up cell."
                )

    # ------------------------------------------------------------------ teardown
    def _arm_callbacks(self) -> None:
        if self._armed:
            return
        self._armed = True
        atexit.register(self.stop)
        for name in ("SIGINT", "SIGTERM", "SIGHUP"):
            sig = getattr(signal, name, None)
            if sig is None:
                continue
            try:
                previous = signal.getsignal(sig)
            except Exception:
                continue

            def handler(signum, frame, _sig=sig, _prev=previous):
                try:
                    signal.signal(_sig, _prev)  # second Ctrl-C interrupts for real
                except Exception:
                    pass
                threading.Thread(
                    target=self.stop, kwargs={"reason": f"signal {signum}"}, daemon=True
                ).start()

            try:
                signal.signal(sig, handler)
            except (ValueError, OSError):
                pass  # not the main thread, or unsupported

    def _restore_concurrency(self) -> None:
        """Put the provider's configured maxConcurrency back; the runtime value is not durable."""
        baseline = self._concurrency_baseline
        if baseline is None:
            log("provider     : no recorded maxConcurrency baseline — leaving the runtime value in place")
            return
        try:
            if self.client.provider_concurrency() == baseline:
                return
            self.client.set_provider_concurrency(baseline)
            log(f"provider     : maxConcurrency restored to {baseline} (re-GET verified)")
        except Exception as exc:
            log(f"WARNING: could not restore maxConcurrency={baseline} ({exc})")

    def _await_teardown(self) -> None:
        """Wait for an in-flight teardown instead of returning while sockets are still closing.

        atexit and the signal handlers can both land here while the heartbeat thread is already
        tearing down; returning early lets the interpreter exit between "rung disabled" and
        "tailscale down", which leaves the daemon joined to no purpose.
        """
        if self._done.wait(timeout=300):
            return
        log("WARNING: teardown is still running after 300 s; returning without waiting further")

    def stop(self, reason: str = "", force: bool = False) -> None:
        with self._guard:
            if self._stopped:
                self._await_teardown()
                return
            self._stopped = True
            self._done.clear()
        log(f"teardown     : start{' — ' + reason if reason else ''}")
        rung_disabled = not getattr(self, "_rung_on", False)
        if getattr(self, "_rung_on", False):
            # 1. rung first: Plexus must stop selecting this node before its sockets die.
            # If it cannot be verified off, do not create a routed dead endpoint.
            for attempt in range(3):
                try:
                    self.client.set_rung_enabled(False)
                    self._rung_on = False
                    rung_disabled = True
                    log(f"rung         : {PROVIDER}:{MODEL} disabled (re-GET verified)")
                    break
                except Exception as exc:
                    log(f"rung         : disable attempt {attempt + 1}/3 failed ({exc})")
                    time.sleep(2.0)
            if not rung_disabled and not force:
                log(
                    "teardown     : INCOMPLETE — rung is still enabled, so workers, proxy and "
                    "tailnet are being LEFT RUNNING to avoid a routed dead endpoint. Fix the "
                    "control-plane error and re-run stop(), or call stop(force=True) to destroy "
                    "the node anyway."
                )
                with self._guard:
                    self._stopped = False
                self._done.set()
                return
            if not rung_disabled:
                log(
                    "WARNING: stop(force=True) is destroying the node while the rung is still "
                    f"enabled; disable it by hand: alias {ALIAS} -> LOCAL -> {PROVIDER}"
                )
        self._stop.set()
        try:
            # 2. provider concurrency back to the configured baseline
            self._restore_concurrency()
            # 3. workers
            for worker in self.workers:
                log(f"worker {worker.slot}     : stopping")
                try:
                    worker.stop()
                except Exception as exc:
                    log(f"worker {worker.slot}     : stop failed ({exc})")
            # 4. proxy
            if self.proxy is not None:
                try:
                    self.proxy.stop()
                    log("proxy        : closed")
                except Exception as exc:
                    log(f"proxy        : stop failed ({exc})")
            # 5. tailnet
            if self.ts is not None:
                try:
                    self.ts.serve_reset()
                except Exception as exc:
                    log(f"tailscale    : serve reset failed ({exc})")
                try:
                    self.ts.down()
                    log("tailscale    : node left the tailnet")
                except Exception as exc:
                    log(f"tailscale    : down failed ({exc})")
                try:
                    device_id = getattr(self, "device_id", None)
                    if device_id and hasattr(self.ts, "delete_device"):
                        if self.ts.delete_device(device_id):
                            log(f"tailscale    : device {device_id} deleted from the tailnet")
                    elif device_id:
                        log("WARNING: this Tailscale client cannot delete the node via the API")
                except Exception as exc:
                    log(f"tailscale    : node delete failed ({exc})")
                try:
                    self.ts.stop_daemon()
                except Exception as exc:
                    log(f"tailscale    : daemon stop failed ({exc})")
        finally:
            self._done.set()
            log("teardown     : done")

    def wait(self) -> None:
        log("runtime      : holding open — Ctrl-C for a graceful shutdown")
        try:
            while not self._stop.wait(1.0):
                pass
        except KeyboardInterrupt:
            log("runtime      : interrupt")
            self.stop(reason="KeyboardInterrupt")
            return
        self._await_teardown()

    # -------------------------------------------------------------------- status
    def status(self) -> None:
        lines = []
        gpus = ", ".join(f"cuda:{gpu['index']} {gpu['name']} {gpu['gib']:.2f} GiB" for gpu in GPUS)
        lines.append(f"gpus        : {len(GPUS)} — {gpus}")
        for worker in self.workers:
            pid = worker.proc.pid if worker.proc is not None else "-"
            kv = worker.ctx * KV_BYTES_PER_TOKEN / 2**30
            lines.append(
                f"worker {worker.slot}     : cuda:{worker.cuda} port={worker.port} ctx={worker.ctx} "
                f"kv={kv:.2f} GiB pid={pid} alive={worker.alive()}"
            )
        if self.gpu_mib:
            evidence = ", ".join(f"slot {slot}={mib} MiB" for slot, mib in sorted(self.gpu_mib.items()))
            lines.append(f"gpu proof   : {evidence}")
        if self.proxy is not None:
            lines.append(
                f"proxy       : 127.0.0.1:{self.proxy.port} served={self.proxy.served} "
                f"failures={self.proxy.failures}"
            )
        lines.append(f"tailnet     : {self.ip or '-'}  {self.dnsname or '-'}")
        lines.append(f"serve       : tcp {self.serve_port} -> 127.0.0.1:{self.serve_port}")
        lines.append(
            f"provider    : {PROVIDER} systemone -> {self.base_url or '-'} "
            f"maxConcurrency={self.concurrency or '-'} timeoutMs={self.provider_timeout_ms or '-'}"
        )
        lines.append(f"rung        : {self.client.describe_rung()}")
        print("=" * 78)
        for line in lines:
            print(line)
        print("=" * 78, flush=True)

In [ ]:
# [11/13] bring the node up and register it. Prints the status block on success.
RUNTIME = Runtime()
RUNTIME.start()

In [ ]:
# [12/13] hold the runtime open. Keep the Colab tab open: closing it ends the VM.
# Ctrl-C here disables the rung, kills the workers and leaves the tailnet.
RUNTIME.wait()

In [ ]:
# [13/13] explicit graceful shutdown — idempotent, safe to run twice.
# Order: rung disabled (verified) -> workers SIGTERM/SIGKILL -> proxy -> tailscale down.
RUNTIME.stop(reason="manual shutdown cell")
print("rung now:", RUNTIME.client.describe_rung())

## Wire contract (what Plexus sends, what llama.cpp must answer)

Plexus POSTs to `{provider.api_base_url.systemone}/systemone` with
`Authorization: Bearer <provider api_key>` and a `decisions` body:

```json
{"model": "clef-flash",
 "state": "<string | object | array>",
 "questions": {
   "<id>": {"type": "noul", "instructions": "<string | object | array>"},
   "<id>": {"type": "choice", "instructions": "...", "criteria": {"<option>": "<description | null>"}},
   "<id>": {"type": "score", "instructions": "...", "criteria": ["<level description>", "..."]}
 }}
```

It requires at least this back:

```json
{"model": "<string>",
 "answers": {
   "<id>": {"type": "choice", "choice": "<option>", "probabilities": {}, "confidence": 0.0},
   "<id>": {"type": "score", "score": 0.0, "legend": {}, "probabilities": {}},
   "<id>": {"type": "noul", "noul": 0.0}
 },
 "usage": {"input_tokens": 0, "output_tokens": 0}}
```

`llama-server` implements exactly this at `POST /v1/systemone`: it reads the GGUF's named
`systemone` chat template and the decision head, and it does **not** validate the `model` field —
so the routed model string stays `clef-flash`. A real call through Plexus returned choice/score/
noul answers with `usage {"input_tokens": 335, "output_tokens": 0}`.

The proxy passes the path and body through unchanged, so the contract is end-to-end: proxy auth
(`Bearer <CLEF_REMOTE_KEY>`) → worker auth (same key) → llama.cpp `/v1/systemone`.

**Timeouts.** The provider is created with `timeoutMs=600000` (10 min); the proxy's upstream
timeout is 540 s, so its own 502 reaches Plexus before Plexus abandons the request. Ten minutes is
still not a promise for large prompts: pre-fill on one T4 runs in the low hundreds of tokens per
second, so a ~100k-token `state` costs minutes of pre-fill before the first token, and a big
`questions` block adds more. Keep `state` well inside the worker's context window and measure
`usage.input_tokens` on a real call before routing long prompts here.

## Verifying from another host

From a tailnet peer (the Plexus host or the operator workstation). `100.x.y.z` is the node's tailnet IPv4, printed by
the status block in the bring-up cell.

```bash
# 1. node liveness (unauthenticated by design: counters + per-worker health)
curl -s http://100.x.y.z:8080/health | jq

# 2. the proxy refuses unauthenticated inference
curl -s -o /dev/null -w '%{http_code}\n' -X POST http://100.x.y.z:8080/v1/systemone -d '{}'
# -> 401

# 3. a direct decisions call at the worker
curl -s http://100.x.y.z:8080/v1/systemone \
  -H "Authorization: Bearer $CLEF_REMOTE_KEY" -H 'content-type: application/json' \
  -d '{"model":"clef-flash","state":"ping","questions":{"n":{"type":"noul","instructions":"hi"}}}' | jq

# 4. what Plexus actually stored for the provider
curl -s -H "x-admin-key: $PLEXUS_ADMIN_KEY" \
  https://plexus.example.ts.net/v0/management/providers/clef-llama-remote | jq '.api_base_url'

# 5. rung state in the alias
curl -s -H "x-admin-key: $PLEXUS_ADMIN_KEY" \
  https://plexus.example.ts.net/v0/management/aliases/cloudflare/clef-flash \
  | jq '.target_groups[] | {name, targets}'

# 6. end-to-end through Plexus (the same shape the self-test uses)
curl -s https://plexus.example.ts.net/v1/systemone \
  -H "Authorization: Bearer $PLEXUS_API_KEY" -H 'content-type: application/json' \
  -d '{"model":"clef-flash","state":"ping","questions":{"n":{"type":"noul","instructions":"hi"}}}' | jq
```

Inside the VM the status block is authoritative: worker `n_ctx` is read back from `/props`, and
the provider URL and rung flag are re-`GET` after every write.

## Direct provider calls

Plexus resolves `direct/<provider>/<model>` before the alias selector, so a direct call proves one
provider rather than the whole fallback chain. The notebook's limited key `clef-llama-colab-v2`
allows `clef-llama-remote` and `clef-flash-mac` only; anything else is refused with HTTP 403 rather
than silently rerouted.

```bash
# this notebook's provider, bypassing cloudflare/clef-flash
curl -s https://plexus.example.ts.net/v1/systemone \
  -H "Authorization: Bearer $PLEXUS_API_KEY" -H 'content-type: application/json' \
  -d '{"model":"direct/clef-llama-remote/clef-flash","state":"ping","questions":{"n":{"type":"noul","instructions":"hi"}}}' | jq

# the always-on Mac rung
curl -s https://plexus.example.ts.net/v1/systemone \
  -H "Authorization: Bearer $PLEXUS_API_KEY" -H 'content-type: application/json' \
  -d '{"model":"direct/clef-flash-mac/clef-flash","state":"ping","questions":{"n":{"type":"noul","instructions":"hi"}}}' | jq
```

The end-to-end self-test uses `direct/clef-llama-remote/clef-flash`, so a passing test proves this
rung specifically, not merely that the alias answered from some other provider.


## Troubleshooting

**Worker dies during load.** `RuntimeError: worker N exited rc=... ` carries the tail of
`/content/logs/llama-N.log`. CUDA OOM on a 15 GiB T4 means the context is too greedy for the
card's free memory: re-run the GPU cell with `CLEF_CTX=65536` (or 32768) and bring up again.

**`worker(s) [0] hold less than 2048 MiB on a GPU`.** Bring-up proved the weights are not on the
card, so the rung was never enabled. Either the wrong release was unpacked (no `libggml-cuda.so`
in `llama-b12292/`), `LD_LIBRARY_PATH` is missing the `cudart-*` directory, or CUDA initialisation
failed and llama.cpp loaded on the CPU. The worker's `/content/logs/llama-N.log` names the cause;
re-run [shell 2/3] and the locate cell.

**`tcp/8080 does not answer over the tailnet`.** The reachability probe went out through
`tailscaled`'s userspace SOCKS5 proxy and came back through `tailscale serve`; it failed, so
Plexus was not pointed at this node. Check `tailscale serve status` for `tcp://100.x:8080`, that
`BackendState` is `Running`, and that the ACL lets `tag:endpoints` traffic reach the port. If the
probe itself is the problem (userspace proxy quirks), `CLEF_ALLOW_UNREACHABLE=1` skips the gate and
leaves reachability unproven.

**`n_ctx` comes back smaller than requested.** The server clamped it (VRAM, or `--parallel`).
The warning prints both numbers; lower `CLEF_CTX` until they agree.

**CUDA driver mismatch / `libcudart.so.12: cannot open shared object`.** The `cudart` tarball must
be extracted next to the binaries — that is what [shell 2/3] does — and `LD_LIBRARY_PATH` must
include it, which `Worker.start()` sets from the directories discovered at runtime. If the log
shows a missing `.so`, re-run [shell 2/3] and the locate cell.

**`duplicate Tailscale nodes survived deletion`.** The API token could not delete a device, or a
device re-registered while cleanup was running. Stop any other notebook using this hostname and
re-run the bring-up cell. Service is unaffected in the meantime: Plexus is pointed at the tailnet
IP, never the DNS name (the Plexus host runs with CorpDNS/MagicDNS off).

**Plexus gets 401.** `CLEF_REMOTE_KEY` and the provider's `api_key` disagree. Update one to match
the other; the notebook reads the key from the secrets panel, the provider holds its own copy.

**`tailscale up` prints a login URL instead of joining.** The key minted for this run was rejected,
or `TAILSCALE_API_KEY` lacks device/key write authority. Check the API token's scope; the notebook
mints a single-use key per run and does not keep a standing `TS_AUTHKEY`.

**ACL blocks traffic.** The Plexus host must be allowed to reach this node on the serve port (8080
or 8443). In userspace mode the daemon still terminates inbound connections to served ports, so a
`tag:endpoints` → caller grant on those ports is what matters.

**Plexus keeps failing after a hard VM death / VM deletion.** A deleted or evicted runtime runs
no teardown at all — `atexit` and the signal handlers only exist while the kernel is alive — so the
rung stays enabled and the provider goes into cooldown. Either wait it out or clear the LOCAL
target's `enabled` flag through the management API (`tools/rung.py off`); the provider is
configured with `disable_cooldown: false`, `stall_cooldown: false`.

**90-minute idle disconnect.** Colab drops an idle runtime; the kernel dies mid-teardown and
Plexus is left pointing at a dead IP. Keep the tab open, or accept the cooldown and restart.

**Port ladder.** If 8080 is occupied inside the VM the proxy binds 8443, `tailscale serve` is
published on 8443, and the provider base URL is rewritten with 8443 — no manual step.